# <font color=#003599>PHYS 249 Unit 2: Systems of Linear Equations</font>
> *Last Updated by Brenda Lee on September 30, 2026*

At the end of each unit, a Jupyter notebook will be provided that summarizes all of the key concepts learned in the entire unit. Please save these notebooks because they'll come in handy for future courses and provide you with a quick reference for both linear algebra and computational methods. Along the way, maybe you can learn some LaTeX as well (you can open up my markdown cells and see how I wrote the code for the equations), though this is certainly not part of the course.

## <font color=#0072DA>Topic 9: Applications of Systems of Linear Equations with Computational Methods and Visualization with Python</font>

From Topics 7-8, we learnt about how to solve systems of linear equations using multiple techniques. Topic 9 will explore how to solve systems of linear equations and apply them to physics problems from various fields using Python code.


---

### <font color=#0072DA>9.1 Solving Systems of Linear Equations</font>
#### <font color=#003599>Key Concepts </font>
A **linear equation** is an equation that can be written in the form of $a_1x_1 + a_2x_2 + ... + a_nx_n = b$ such that the *a*'s are **coefficients** of the equation while $x_i$ are unknown to be solved for. A **system of linear equations** is then a collection of several linear equations that can have a **solution**, which is a set of numbers that coincide with the solved values of $x_i$.

A **solution set** is a collection of possible solutions, while a system of linear equations with at least one solution being called **consistent**.

If two separate systems of linear equations have the same solution set, they are said to be **equivalent**.

We can represent a system of linear equations as an **augmented matrix**, written as:

$\left[\begin{array}{cc|c} 1 & 2 & 3 \\ 4 & 5 & 6 \\\end{array} \right]$

<div class="alert alert-block alert-info"> &#x1F4A1; <b>GAUSSIAN ELIMINATION WITH BACK-SUBSTITUTION</b> is when we solve a system of linear equations in a systematic manner by applying operations such as scalar multiplication of a row, interchange of two rows, or addition of multiples of rows such that each equation has a <b>leading variable</b> that doesn't appear in the row below it. When writing these out as augmented matrices, we note that these three operations are called <b>elementary row operations or EROs</b> and that the process of simplifying these equations is called <b>row reduction</b>. The final matrix, when properly row-reduced, is said to be in <b>REF or row echelon form</b>, where the leading variable has a coefficient of 1 and all that come below it are 0's.</div>

<div class="alert alert-block alert-info"> &#x1F4A1; <b>GAUSS-JORDAN ELIMINATION</b>, also called Complete Eliminiation, is when the leading variable in the <i>j</i>th equation has been eliminated from every other equation. A matrix corresponding to these coefficients is said to be in <b>RREF or row-reduced echelon form</b>, where the leading variable has a coefficient of 1 and all that come above or below it are 0's.</div>

<div class="alert alert-block alert-info"> &#x1F4A1; <b>RANK OF MATRIX</b> The rank of a matrix A is the number of leading ones in its RREF form and is denoted by $\text{rank}(A)$. The <b>System-Rank Theorem</b> states that if $\left[\begin{array}{c|c} A & \vec{b} \end{array} \right]$ is a system of $m$ linear equations in $n$ variables, then the number of parameters in the general solution is equal to $n-\text{rank}(A)$</div>

Some additional terminology that may be useful:

- A linear equation is **homogeneous** if the right-hand side is zero (a system of linear equations is called this is ALL right-hand sides are zero)
- We call $\vec{0}$ the **trivial solution**
- The solution set of a homogeneous system is called the **solution space** of the system

#### <font color=#8B0000>Computational Methods with Python</font>

Let's see if there are ways to solve systems of linear equations with Python!

In [1]:
# Let's try a manual way of row-reducing matrices in Python
import numpy as np

def input_matrix():
    # Get matrix dimensions from user
    rows = int(input("Enter the number of rows: "))
    cols = int(input("Enter the number of columns: "))

    #Initialize an empty matrix and get matrix values from user
    customMatrix = []
    print("Enter the matrix values row by row (space-separated):")
    for i in range(rows):
        row = list(map(float, input(f"Row {i + 1}: ").strip().split()))
        # Check if the number of values in the row matches the expected number of columns
        if len(row) != cols:
            print(f"Error: You must enter exactly {cols} values for this row")
            raise ValueError(f"Expected {cols} values for row {i + 1}, but got {len(row)}.")
        customMatrix.append(row) # puts each value in its spot in matrix
    return customMatrix

def matrix_rref(matrix):
    """
    Gaussian elimination and back-substitution with recursion to complete elimination
    Param: matrix --> array with float values
    Result: matrix in RREF
    """
    #Convert input matrix to a numpy array with float values
    #Allows us to divide properlu
    matrix = np.array(matrix, dtype=float)

    #Get number of rows and columns
    rows, cols = matrix.shape

    #Gaussian elimination for loop to create an upper triangular matrix
    for i in range(min(rows, cols)):
        # Find the pivot row (with first and largest leading variable)
        #The max() function finds the row index with largest absolute value in the current column (i) starting from row i
        #lambda is the lambda function that takes row index r and returns the absolute value of the element in column i of that row
        #returns the absolute valueq
        #the key in max() tells the function how to compare elements
        #which is absolute values (abs)
        #specifying key-lambda r: abs(matrix[r, i]) ensures that max() selects the row with the largest abs value in the current column
        maxVal_row = max(range(i, rows), key=lambda r: abs(matrix[r, i]))

        # Swap the current row with the pivot row if needed
        #This brings the row with the largest pivot element to the top
        matrix[[i, maxVal_row]] = matrix[[maxVal_row, i]]

        #we normalize the pivot row (ith row) by dividing the whole row by the pivot element
        #this makes the pivot element in the ith column equal to 1
        pivot = matrix[i, i]
        #Python equality operator to check if two values are equal
        #then it spits out true or false boolean term
        #true when values are equal
        if pivot == 0: #Ensure we dont diqvide by zero
            continue  # Skip this column if the pivot is zero (no leading variable in this column)
        matrix[i] = matrix[i] / pivot

        # We then eliminate all the elements below pivot in the same col
        # turn the column below pivot into zeros
        #the loop starts from row directly below pivot row (i)
        #all the way to rows-1(exclusive of last point)
        for j in range(i + 1, rows):
            matrix[j] = matrix[j] - matrix[i] * matrix[j, i]
            #Subtract the appropriate multiple of the pivot row from the jth row
        
    # Back-substitution to eliminate elements above the pivots
    for i in range(min(rows, cols) - 1, -1, -1):
        pivot = matrix[i, i]
        if pivot != 0: # check for non zero pivot
            #we want to eliminate all elements above the pivot in the same column
            for j in range(i - 1, -1, -1):
                matrix[j] = matrix[j] - matrix[i] * matrix[j, i]
                #Subtract the appropriate multiple of the pivot row from the jth row
    #optional cleanup (great for significance checks)
    matrix[np.abs(matrix) < 1e-10] = 0  # Set very small values to zero

    #Return final rref form of matrix
    return matrix
        
# Test with a matrix 
userMatrix = input_matrix()
np.set_printoptions(precision = 3, floatmode = 'fixed') #formatting for printing
if userMatrix is not None:
    print ("The matrix you entered is:")
    for row in userMatrix:
        print(row)
    print ("The RREF of the matrix is:")
    for row in matrix_rref(userMatrix):
        print(row)
    

Enter the matrix values row by row (space-separated):
The matrix you entered is:
[1.0, 1.0, -2.0, 4.0]
[1.0, 3.0, -1.0, 7.0]
[2.0, 1.0, -5.0, 7.0]
The RREF of the matrix is:
[1.000 0.000 0.000 0.000]
[0.000 1.000 0.000 2.000]
[ 0.000  0.000  1.000 -1.000]


**Documentation Link for `range`**: https://docs.python.org/3/library/functions.html#func-range

**Documentation Link for `array`**: https://numpy.org/doc/stable/reference/generated/numpy.array.html

**Documentation Link for `input`**: https://docs.python.org/3/library/functions.html#input

**Documentation Link for `max`**: https://docs.python.org/3/library/functions.html#max

In [2]:
# Sometimes, it's easier to use readily available functions
# There is one in SymPy library we can use!
import sympy as sp
#define matrix with user input from before
userMatrix = sp.Matrix((input_matrix()))

#Perform RREF row reduction
rref_matrix, pivot_columns = userMatrix.rref()
print("The RREF of the matrix is:")
print(rref_matrix)

#Prettier way to present all this
sp.pprint(rref_matrix)

Enter the matrix values row by row (space-separated):
The RREF of the matrix is:
Matrix([[1, 0], [0, 1]])
⎡1  0⎤
⎢    ⎥
⎣0  1⎦


In [3]:
# How about doing this in LaTeX output?
# SymPy offers a package for this type of display
from sympy import init_printing
init_printing(use_latex="mathjax")
print ("The RREF of the matrix in latex is:")
sp.pprint(rref_matrix)

The RREF of the matrix in latex is:
⎡1  0⎤
⎢    ⎥
⎣0  1⎦


**Documentation Link for `sp.Matrix`:** https://docs.sympy.org/latest/modules/matrices/matrices.html

**Documentation Link for `matrix.rref`:** https://docs.sympy.org/latest/modules/matrices/matrices.html#sympy.matrices.matrixbase.MatrixBase.rref

**Documentation Link for `SymPy Printing`:** https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html

In [4]:
# We can get an RREF form, but what is the final solution? We can read it off of the matrix but we can also skip the whole RREF
# And just solve using np.linalg.solve(A, b) for an augmented matrix
A = np.array([[1, 1, 0], [2, 4, 1], [1, 2, 1]])
B = np.array([-7, -16, 9])

#We can solve directly for the system of equations Ax = B using np.linalg.solve
solution = np.linalg.solve(A, B)
print("The solution to the system of equations is:")
print("x1 = {:.3f}, x2 = {:.3f}, x3 = {:.3f}".format(*solution))

The solution to the system of equations is:
x1 = 11.000, x2 = -18.000, x3 = 34.000


In [5]:
# Sometimes, we want to check the solution to be safe
checkSolution = np.dot(A, solution)
print("Computed b from A*x:")
print(checkSolution)

Computed b from A*x:
[ -7.000 -16.000   9.000]


**Documentation Link for `np.linalg.solve`:** https://numpy.org/doc/2.0/reference/generated/numpy.linalg.solve.html

### <font color=#0072DA>9.2 Applications in Physics with Computational Methods</font>

Systems of linear equations show up quite often in the physical sciences. We'll explore a few applications below, and utilize some of the code we've created from before to solve application questions.

#### <font color=#8B0000>9.2.1 Spring-Mass Systems</font>

Let's consider a spring-mass system with three springs as shown in the figure below.

<img src="PHYS249_Topic09Fig1.jpg" alt="Spring-Mass System" style="width:auto;height:200px;">
According to Hooke's Law where $F=-kx$ ($F$ is the force required to stretch/compress a spring with spring constant $k$ by a distance $x$), we know that we can create two equations from this to solve for the system in equilibrium. Looking at the first mass $m_1$, it is being acted upon by two springs $k_1, k_2$. If we are stretching $m_1$ to the right, the restoring force acting on $m_1$ will be to the left by $k_1$ while it is stretched by $x_1$, while $k_2$ acts on mass $m_1$ through a distance $x_2-x_1$. We can then say that in equilibrium, the sum of the forces acting on $m_1$ must be $0$:

$$F_1 = -k_1x_1 + k_2(x_2-x_1) \rightarrow F_1 -k_1x_1 + k_2(x_2-x_1) = 0$$

Mass $m_2$ is being acted on by two springs: $k_2, k_3$. We can then create another equation for equilibrium for $m_2$:

$$F_2 = -k_3x_2 - k_2(x_2-x_1) \rightarrow F_2 -k_3x_2 - k_2(x_2-x_1) = 0$$

Now, we can solve for this if we add in some values. Let's say that $k_1 = 2$ N/m, $k_2 = 4$ N/m, and $k_3 = 3$ N/m. Find the equilibrium displacements $x_1, x_2$ if forces $F_1 = 10$ N and $F_2 = 5$ N are applied to masses $m_1, m_2$ respectively.

We can rearrange the two equations so we get: 
$$(k_1+k_2)x_1 -k_2x_2 = F_1$$
$$-k_2x_1 + (k_2+k_3)x_2 = F_2$$


In [6]:
# Let's set up an augmented matrix to solve. 
import numpy as np
# set variables
k1 ,k2, k3 = 2, 4, 3 # Units of N/m
f1, f2 = 10, 5 #Units of N

A = np.array([[k1+k2, -k2],
              [-k2, k2+k3]])
B = np.array([f1, f2])

# solve directly for A*x=B
solution = np.linalg.solve(A, B)
print("The solution to the system of equations is:")
print("x1 = ", solution[0], "m, x2 = ", solution[1], "m")

#Check solution 
checkSolution = np.dot(A, solution)
if np.allclose(checkSolution, B):
    print("The solution is verified: A*x = B")
else:
    print("The solution is NOT verified: A*x != B, please check code")

The solution to the system of equations is:
x1 =  3.461538461538461 m, x2 =  2.6923076923076916 m
The solution is verified: A*x = B


**Documentation Link for `np.allclose`:** https://numpy.org/doc/stable/reference/generated/numpy.allclose.html

#### <font color=#8B0000>9.2.2 Electrical Circuits and Kirchhoff's Laws</font>

In Electricity & Magnetism, you would have covered the flow of electrical current in simple electrical circuits. In an electrical circuit, the **current I** has a direction and therefore has a sign attached to it. **Voltage V** is a signed quantity, while **resistance R** is a positive scalar quantity. To relate these quantities, **Ohm's Law** is given by $V = IR$. We then have **Kirchhoff's Laws** that help determine their behaviour.

**Kirchhoff's Current Law** states that at any node or junction where several currents enter, the signed sum of the currents entering the node is zero. In other words, we have $I_{\text{tot}} = 0$ at a singular node.

**Kirchhoff's Voltage Law** states that in a closed loop that consists of only resistors and an **electromotive force or emf $\varepsilon$**, the sum of the voltage drops across resistors is equal to $\varepsilon$. In other words, $\varepsilon = V_{\text{tot}}$

Let's consider this resistor circuit below:

<img src="PHYS249_Topic09Fig2.jpg" alt="Example Resistor Circuit" style="width:auto;height:250px;">

Let's assume we are given the following values:

- $\varepsilon_1=8$ V and $\varepsilon_2=10$ V
- $R_1 = 1 \Omega, R_2 = 2 \Omega, R_3 = 3 \Omega, R_4 = 4 \Omega, R_5 = 2 \Omega, R_6 = 2 \Omega, R_7 = 3 \Omega,  R_8 = 1 \Omega$

And we are now asked to solve for the current $I$ in each loop. Apply Kirchhoff's Voltage Law, where $\varepsilon = V_{\text{tot}}$ and $V = IR$
$$\text{Top Left Loop:}\qquad\varepsilon_1 = I_1R_1 + (I_1-I_2)R_2 + (I_1-I_3)R_4$$
$$\text{Top Right Loop:}\qquad\varepsilon_2 = I_2R_3 + (I_2-I_4)R_5 + (I_2-I_1)R_2$$
$$\text{Bottom Left Loop:}\qquad 0 = I_3R_6 + (I_3-I_1)R_4 + (I_3-I_4)R_7$$
$$\text{Bottom Right Loop:}\qquad -\varepsilon_2 = I_4R_8 + (I_4-I_3)R_7 + (I_4-I_2)R_5$$

We should rewrite in a form that's easy to convert to an augmented matrix to solve for $I$. But why don't we use code for that to save work?

In [7]:
# Sympy has a neat tool for us to use to extra coefficients for our desired variables
from sympy import symbols, Eq, solve, pprint
e1, e2 = 8, 10
r1, r2, r3, r4, r5, r6, r7, r8 = 1, 2, 3, 4, 2, 2, 3, 1
i1, i2, i3, i4 = symbols('i1 i2 i3 i4') # NO COMMAS!
# we define system of LEs
eq1 = Eq(i1*r1 + (i1-i2)*r2 + (i1-i3)*r4, e1)
eq2 = Eq(i2*r3 + (i2-i4)*r5 + (i2-i1)*r2, e2)
eq3 = Eq(i3*r6 + (i3-i1)*r4 + (i3-i4)*r7, 0)
eq4 = Eq(i4*r8 + (i4-i3)*r7 + (i4-i2)*r5, -e2)

solution = solve((eq1, eq2, eq3, eq4), (i1, i2, i3, i4))
print("The solution to the system of equations is:")
pprint(solution)

#manual check requires subs function
check_eq1 = eq1.subs(solution)
check_eq2 = eq2.subs(solution)
check_eq3 = eq3.subs(solution)
check_eq4 = eq4.subs(solution)
print("Check equations after substituting solution:")
pprint(f"Eq1: {check_eq1.simplify()}")
pprint(f"Eq2: {check_eq2.simplify()}")
pprint(f"Eq3: {check_eq3.simplify()}")
pprint(f"Eq4: {check_eq4.simplify()}")

The solution to the system of equations is:
⎧    2172      1922      702       -790 ⎫
⎨i₁: ────, i₂: ────, i₃: ────, i₄: ─────⎬
⎩    1069      1069      1069      1069 ⎭
Check equations after substituting solution:
Eq1: True
Eq2: True
Eq3: True
Eq4: True


In [8]:
# If you REALLY want to solve with NumPy, then we need to evaluate the coefficients numerically
# We would extract our coefficients for each variable from the equations
# This is useful if you want to rewrite the equations in variables i1, i2, i3, i4 separately
coeff_eq1 = [eq1.lhs.coeff(var) for var in (i1, i2, i3, i4)]
coeff_eq2 = [eq2.lhs.coeff(var) for var in (i1, i2, i3, i4)]
coeff_eq3 = [eq3.lhs.coeff(var) for var in (i1, i2, i3, i4)]
coeff_eq4 = [eq4.lhs.coeff(var) for var in (i1, i2, i3, i4)]
print([coeff_eq1, coeff_eq2, coeff_eq3, coeff_eq4])

# Set up your coefficient matrix A and the right hand side b
A = np.array([coeff_eq1, coeff_eq2, coeff_eq3, coeff_eq4], dtype=float)
b = np.array([e1, e2, 0, -e2], dtype=float)
solution = np.linalg.solve(A,b)
print(f"The solution is given by: i1 = {solution[0]} Ω, i2 = {solution[1]} Ω, i3 = {solution[2]} Ω, i4 = {solution[3]} Ω.")

# Then do a check
checkSolution = np.dot(A, solution)
if np.allclose(checkSolution,b):
    print("Solution is correct as A*x = b")
else:
    print("Solution is not correct. Please check code.")

[[7, -2, -4, 0], [-2, 7, 0, -2], [-4, 0, 9, -3], [0, -2, -3, 6]]
The solution is given by: i1 = 2.031805425631431 Ω, i2 = 1.7979420018709071 Ω, i3 = 0.656688493919551 Ω, i4 = -0.7390084190832554 Ω.
Solution is correct as A*x = b


#### <font color=#8B0000>9.2.3 Balancing Chemical Equations</font>

If you have taken a chemistry class, you would have had to balanced chemical equations. Let's look at one example here:
$$H_3PO_4 + Mg(OH)_2 \rightarrow Mg_3(PO_4)_2 + H_2O$$

Since we want to balance this, we essentially find constants $x_1, x_2, x_3, x_4$ such that:
$$x_1H_3PO_4 + x_2Mg(OH)_2 \rightarrow x_3Mg_3(PO_4)_2 + x_4H_2O$$

But this isn't a vector equation. We need variables. So we can represent the molecules $H, P, O, M$ in the equation with a set of vectors in $\mathbb{R}^4$:

$$x_1\begin{bmatrix} 3\\1\\4\\0 \end{bmatrix} + x_2\begin{bmatrix} 2\\0\\2\\1 \end{bmatrix} = x_3\begin{bmatrix} 0\\2\\8\\3 \end{bmatrix} + x_4\begin{bmatrix} 2\\0\\1\\0 \end{bmatrix}$$

Which can be rewritten as:

$$x_1\begin{bmatrix} 3\\1\\4\\0 \end{bmatrix} + x_2\begin{bmatrix} 2\\0\\2\\1 \end{bmatrix} - x_3\begin{bmatrix} 0\\2\\8\\3 \end{bmatrix} - x_4\begin{bmatrix} 2\\0\\1\\0 \end{bmatrix} = \vec{0}$$

Equivalently, in matrix form:

$$
\begin{bmatrix}
3 & 2 & 0 & -2 \\
1 & 0 & -2 & 0 \\
4 & 2 & -8 & -1 \\
0 & 1 & -3 & 0
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3 \\
x_4
\end{bmatrix}
=
\begin{bmatrix}
0 \\
0 \\
0 \\
0
\end{bmatrix}
$$

We can now solve this for the coefficients!

In [9]:
from sympy import symbols, Eq, solve, pprint

#define variables and equations in order of H, P, O, M
x1, x2, x3, x4 = symbols('x1 x2 x3 x4') # NO COMMAS!
eq1 = Eq(3*x1 + 2*x2 +0*x3 -2*x4, 0)
eq2 = Eq(1*x1 + 0*x2 -2*x3 + 0*x4, 0)
eq3 = Eq(4*x1 +2*x2 -8*x3 -1*x4, 0)
eq4 = Eq(0*x1 +1*x2 -3*x3 + 0*x4, 0)

# solve using sympy
solution = solve((eq1, eq2, eq3, eq4), (x1, x2, x3, x4))
print("The solution to the system of equations is:")
pprint(solution)

The solution to the system of equations is:
⎧    x₄      x₄      x₄⎫
⎨x₁: ──, x₂: ──, x₃: ──⎬
⎩    3       2       6 ⎭


In [10]:
# Now, that's not quite right for balancing equations. We want integer values and x4 is missing
from sympy import denom, lcm, simplify
denom_x1 = denom(solution[x1])
denom_x2 = denom(solution[x2])
denom_x3 = denom(solution[x3])

#to find x4, we should find lcm
x4_lcm = lcm(denom_x1, denom_x2, denom_x3)
print (f'the smallest x4 valuue is {x4_lcm}')

#Sub x4 into the solution and simplify it to reduce to integer values
x1Val = simplify(solution[x1].subs(x4,x4_lcm))
x2Val = simplify(solution[x2].subs(x4,x4_lcm))
x3Val = simplify(solution[x3].subs(x4,x4_lcm))
print (f'x1 = {x1Val} x2 = {x2Val} x3 = {x3Val} x4 = {x4_lcm}')

the smallest x4 valuue is 6
x1 = 2 x2 = 3 x3 = 1 x4 = 6


Therefore, the balanced equation is:
$$2 H_3PO_4 + 3Mg(OH)_2 \rightarrow Mg_3(PO_4)_2 + 6H_2O$$

Independent Computational Practice  
C1
$2x_1+x_2−x_3=8$,  
$−3x_1−x_2+2x_3=−11$,   
$−2x_1+x_2+2x_3=−3$

In [13]:
import numpy as np
#a
A = np.array([[2, 1, -1],
             [-3, -1, 2],
             [-2, 1, 2]])
b = np.array([8, -11, -3])
#b
solution = np.linalg.solve(A,b)
print ("x1, x2, x3", solution)
# Then do a check
#c
checkSolution = np.dot(A, solution)
print (checkSolution)
#d
if np.allclose(checkSolution, b):
    print("Solution is correct as A*x = b")
else:
    print("Solution is not correct. Please check code.")


x1, x2, x3 [ 2.000  3.000 -1.000]
[  8.000 -11.000  -3.000]
Solution is correct as A*x = b


C2 
np.allclose returns True if both arguments are the same array, and the dot product between a and the solution should return b, since Ax=b. Checking the solution is useful because there may be truncation/rounding errors in the solution.